# Project 2: Conduction
## Spring 2026
**Course:** Next-Generation Data Centers & Energy Management  

Name: <First Name, Last Name>
E-mail: email@berkeley.edu 

Academic Integrity & AI Policy
This project is designed to test your ability to write and solve the spatial heat equation. The use of Generative AI (e.g., ChatGPT, Claude, Gemini) to derive equations or write the analysis is strictly prohibited. The goal is for you to grapple with the formulas. All work must be your own original derivation and code.

**Objectives:** Perform conduction analysis in multiple dimensions, and understand the motivation and workings of the lumped capacitance assumption.

### Project Overview
Having modeled the overall data center in Project 1, we will now take a closer look at the heat transfer principles at play in the lower abstraction levels of energy management systems. 
In this project, we will focus on conduction analysis.

We will do 3 problems:
1. 1D conduction analysis; numerical simulation of the 1D heat kernel.
2. 2D conduction analysis of a chip.
3. Lumped capacitance analysis of a cylindrical rod in cross-flow.

Feel free to use any reasonable material properties/constants you see fit, as long as the values and their sources (e.g. MatWeb) are clearly indicated in your submission.
You may choose your own values for numerical quantities that are not explicitly given, e.g. the area/thickness of a chip.

## Problem 1: 1D conduction analysis.

The study of heat transfer revolves around the **heat kernel**, which is a fundamental solution of the heat equation. In simple terms, it shows us how a "unit" or "parcel" of heat, concentrated at one point in space ($x = 0$), spreads out in space over time.
The analytical form of the heat kernel, in 1 dimension, is:
$$ T(x, t) = \frac{1}{2 \sqrt{\pi \alpha t}} \exp \left( - \frac{x^2}{4 \alpha t} \right) $$
where $\alpha = \frac{k}{\rho c_p}$ is the thermal diffusivity of the material.

In this problem, we will numerically simulate the heat kernel using a finite-difference scheme.
Consider a 10-meter-long rod (going from $x \in [-5, 5]$) with a 1 m$^2$ cross-sectional area, made of a material of your choosing.

The finite-difference scheme works as follows:
1. Spatially discretize the rod into 101 lengthwise elements.
2. Start with an initial temperature of 0 K in every element except for the center-most element, which starts with a temperature of 1000K.
3. At each timestep dt = 0.05 sec, use the temperature differences between adjacent elements to compute the heat flux between adjacent elements. That is, use the relation $\dot q = -k A \frac{dT}{dx}$ to compute the heat transfer rate $\dot q$ between each element-element pair. 
4. Using the computed $\dot q$ for each element, update its temperature using $q = \dot q dt = m c_p \delta T \cdot dt$.
5. Repeat steps 3-4 until your desired end-time. Let's say $t_f = 10$ sec for now.

Your task is to implement the scheme and numerically simulate the heat kernel solution using the method described above.

**Deliverable.** Plot your $T(x)$ at least 4 different times, on the same set of axes.

**Deliverable.** On a different set of axes, plot the heat kernel solution $T(x, t)$ at those same times.

**Deliverable.** Compare your numerical results to the analytical results. What do you expect will happen if you decrease dt or increase the element count? Feel free to try it!

**Note.** We recommend using NumPy in your implementation, and Matplotlib for plotting.

In [1]:
#code cell; code solutions go here, including code for plotting.

## Problem 2: 2D conduction analysis.

Having done a relatively theoretical exercise in problem 1, we will turn our attention to chip-cooling.

Suppose you have a 50 mm by 50 mm by 10 mm chip generating 100W of heat, which is equally distributed volumetrically.
The chip is surrounded on 5 sides by a 20mm-thick thermoplastic housing, and coated on its top-side with a 2mm-thick layer of thermal paste. 
The whole system is surrounded on the outside by a constant-temperature medium at 20 $^\circ$C (i.e. this is a Dirichlet boundary condition).
Assume that the whole system starts initially at 20 $^\circ$C, just like the surrounding conditions.

Simulate this conduction problem using a quasi-2D finite difference scheme. That is, the chip, thermal paste layer, and plastic housing should all be modeled as 2D meshes, with thermal interfaces between them.

**Deliverable.** Implement the finite difference scheme and determine the steady-state conditions using appropriate mesh sizing and time-stepping dt.

**Deliverable.** Plot the chip's temperature distribution at steady-state conditions.

**Deliverable.** At the steady state, what percentage of the total heat leaving the chip is exiting through the thermal paste?

In [ ]:
#code cell; code solutions go here, including code for plotting.

## Problem 3: Lumped capacitance analysis.

As you saw in problem 2, modeling the internal temperature variation inside an object is a complex task.
In practice, thermal variations inside an object can be neglected under the **lumped capacitance assumption**, which assumes that the entire body is at the same temperature.
This assumption is only valid if the internal temperature gradient inside the object is small. 
In this problem, we will look more closely at what exactly that means.

Our model for this will be an infinitely long circular steel rod in external cross-flow (i.e. forced convection), with a velocity of 5 m/s. The fluid is air.
Recall that Newton's law of cooling, which we use for convection, is
$$ \dot q = hA \Delta T $$
In general, $h$ is a function of material geometry, fluid properties, and flow conditions.
We will focus more on convection in Project 3. For now, we will provide a function for computing $h$ as a function of the rod diameter $R$ and flow velocity $u$. 
The function is based on a Nusselt correlation, but we don't have to worry about that just yet.

In [1]:
def h(r, u):
    # r is rod outer diameter (mm), u is flow velocity (m/s)
    r = r / 1000             # convert to meters
    nu = 15.89 * 1e-6        # kinematic viscosity, units m^2 / s
    Re = 2 * r * u / nu      # Reynolds number, dimensionless
    Pr = 0.71                # Prandtl number, dimensionless
    Nusselt = 0.3 + (0.62 * Re**0.5 * Pr**(1/3)) * (1 + (0.4/Pr)**(2/3))**(-1/4) * (1 + (Re/282000)**(5/8))**(4/5) 
                             # Churchill and Bernstein correlation
    k_air = 26.3 * 1e-3      # thermal conductivity of air, units W/(m * K)
    h = Nusselt * k_air / (2*r)   # find h from definition of Nusselt number
    return h

Since this conduction/convection problem is radially symmetric, we can compute the thermal resistances between the rod's core temperature $T_c$ and the temperature at infinity $T_\infty$.
We can assume for now that $T_\infty = 20 K$ and $T_c = 40 K$.

**Deliverable.** Using the thermal circuit method, plot the temperature at the outer surface of the rod as a function of rod diameter $r$. Try a range of values from 1 mm to 100 mm, or even larger if you'd like.

In [ ]:
#code cell; code solutions go here, including code for plotting.

We can see that, for small radii, the temperature variation between the rod's core and the rod's outer surface is small.
The Biot number is defined as
$$ Bi = \frac{hD}{k_s}, $$
where $k_s$ is the thermal conductivity of the solid rod.

**Deliverable.** Compute and plot the Biot number as a function of rod radius $r$.

In [ ]:
#code cell; code solutions go here, including code for plotting.

**Deliverable.** What do we notice about the Biot number when the rod radius is small?

(Your answer here)

The Biot number represents the ratio between the thermal resistance due to convection and the thermal resistance due to conduction. In other words, it compares the conductive heat transfer to the convective heat transfer. 
Generally speaking, the lumped capacitance assumption for the rod (i.e. we assume no radial temperature variation inside the rod) is valid when $Bi < 0.1$.

**Deliverable.** Why does that make sense?

(Your answer here)

~~~~ 
THIS IS THE LAST CELL, ANYTHING AFTER THIS WILL NOT BE GRADED. PLEASE DO NOT DELETE. 
~~~~